In [1]:
#include <iostream>
#include <fstream>
#include <vector>
#include <TGraph.h>
#include <TCanvas.h>
#include <string>
#include <cmath>
#include <array>
#include <algorithm>

In [2]:
const char* filename = "chimes.wav";
std::ifstream file(filename, std::ios::binary);
if (!file) { std::cout << "Error: Could not open chimes.wav\n"; return; }

file.seekg(44, std::ios::beg);   // skip standard WAV header

const int N = 65536;             // must be power of 2
std::vector<double> samples;
samples.reserve(N);

short sample_val;
for (int i = 0; i < N && file.read(reinterpret_cast<char*>(&sample_val), sizeof(short)); ++i)
    samples.push_back(static_cast<double>(sample_val));

auto c_audio = new TCanvas("c_audio", "Raw Audio", 800, 400);
TH1D *h_raw = new TH1D("h_raw", "Original Audio (Time Domain);Sample;Amplitude", N, 0, N);
for (int i = 0; i < (int)samples.size(); ++i)
    h_raw->SetBinContent(i+1, samples[i]);
h_raw->Draw();
c_audio->Draw();

In [3]:
TVirtualFFT *fft = TVirtualFFT::FFT(1, const_cast<int*>(&N), "R2C M K");
fft->SetPoints(samples.data());
fft->Transform();

// Extract complex output into our own arrays so we can modify them safely
std::vector<double> re_arr(N/2 + 1), im_arr(N/2 + 1);
for (int i = 0; i <= N/2; ++i)
    fft->GetPointComplex(i, re_arr[i], im_arr[i]);

auto c_spec = new TCanvas("c_spec", "Frequency Spectrum", 800, 400);
// Label x-axis in Hz assuming 44100 Hz sample rate
double sampleRate = 44100.0;
TH1D *h_spec = new TH1D("h_spec", "Frequency Spectrum;Frequency (Hz);Magnitude",
                         N/2, 0, sampleRate / 2.0);

for (int i = 0; i < N/2; ++i) {
    double mag = std::sqrt(re_arr[i]*re_arr[i] + im_arr[i]*im_arr[i]);
    h_spec->SetBinContent(i+1, mag);
}
h_spec->Draw();
c_spec->Draw();

In [4]:
// Set your frequency cutoff range to DISCARD (in Hz):
double freq_lo =  20000.0;    // <── lower bound of rejected band (Hz)
double freq_hi =  22000.0;  // <── upper bound of rejected band (Hz)
double sampleRate = 44100.0;

In [5]:
// Work on copies of the FFT arrays from Cell 2
std::vector<double> re_filt(re_arr), im_filt(im_arr);

for (int i = 0; i <= N/2; ++i) {
    double freq = i * sampleRate / N;          // bin → Hz
    if (freq >= freq_lo && freq <= freq_hi) {  // zero out the unwanted band
        re_filt[i] = 0.0;
        im_filt[i] = 0.0;
    }
}

// ── Inverse FFT ──────────────────────────────────────────────────────────────
TVirtualFFT *fft_back = TVirtualFFT::FFT(1, const_cast<int*>(&N), "C2R M K");
fft_back->SetPointsComplex(re_filt.data(), im_filt.data());
fft_back->Transform();

// Copy IFFT output into our own array (do NOT pass the internal pointer to Adopt)
std::vector<double> cleaned(N);
for (int i = 0; i < N; ++i)
    cleaned[i] = fft_back->GetPointReal(i) / N;   // FFTW output needs /N normalisation

// ── Plot cleaned waveform ─────────────────────────────────────────────────────
auto c_clean = new TCanvas("c_clean", "Cleaned Audio", 800, 400);
TH1D *h_cleaned = new TH1D("h_cleaned",
    "Cleaned Audio (Time Domain);Sample;Amplitude", N, 0, N);
for (int i = 0; i < N; ++i)
    h_cleaned->SetBinContent(i+1, cleaned[i]);
h_cleaned->Draw();
c_clean->Draw();

// ── Write output WAV ──────────────────────────────────────────────────────────
// Find peak for normalisation (preserve original loudness)
double peak = 0;
for (double v : cleaned) if (std::abs(v) > peak) peak = std::abs(v);
double origPeak = 0;
for (double v : samples) if (std::abs(v) > origPeak) origPeak = std::abs(v);
double scale = (peak > 0) ? origPeak / peak : 1.0;

std::vector<short> out_samples(N);
for (int i = 0; i < N; ++i) {
    double val = cleaned[i] * scale;
    // Clamp to 16-bit range
    val = std::max(-32768.0, std::min(32767.0, val));
    out_samples[i] = static_cast<short>(val);
}

// Build a minimal 44-byte PCM WAV header
const char* outfile = "chimes_processed.wav";
std::ofstream wav(outfile, std::ios::binary);

int   dataSize   = N * sizeof(short);
int   chunkSize  = 36 + dataSize;
short audioFmt   = 1;          // PCM
short numChan    = 1;          // mono
int   byteRate   = (int)sampleRate * 1 * 2;
short blockAlign = 1 * 2;
short bitsPerSmp = 16;

wav.write("RIFF", 4);
wav.write(reinterpret_cast<char*>(&chunkSize),  4);
wav.write("WAVE", 4);
wav.write("fmt ", 4);
int subchunk1 = 16;
wav.write(reinterpret_cast<char*>(&subchunk1),  4);
wav.write(reinterpret_cast<char*>(&audioFmt),   2);
wav.write(reinterpret_cast<char*>(&numChan),    2);
int sr = (int)sampleRate;
wav.write(reinterpret_cast<char*>(&sr),         4);
wav.write(reinterpret_cast<char*>(&byteRate),   4);
wav.write(reinterpret_cast<char*>(&blockAlign), 2);
wav.write(reinterpret_cast<char*>(&bitsPerSmp), 2);
wav.write("data", 4);
wav.write(reinterpret_cast<char*>(&dataSize),   4);
wav.write(reinterpret_cast<char*>(out_samples.data()), dataSize);
wav.close();

std::cout << "Saved: " << outfile
          << "  (" << N << " samples, band " << freq_lo
          << "–" << freq_hi << " Hz removed)\n";

Saved: chimes_processed.wav  (65536 samples, band 20000–22000 Hz removed)
